# Argo download and GLORYS12 pairing (data prep for Figures S2 and S5)

Replaces the legacy three-step pipeline in `ROVER/code/FigS3_GLORYS12_verification_using_ARGO.ipynb`
(download → polygon mask → pairing), which produced `argo_glorys_paired_profiles.nc`.

**Bugs fixed relative to the legacy pipeline**

| # | Legacy behaviour | Fix here |
|---|---|---|
| 1 | Yearly argopy request used `date_max = 'YYYY-12'`, an exclusive bound at 1 December → **every December dropped** | Request `[YYYY-01-01, (YYYY+1)-01-01)`, capped at the day after `DATA_END_DATE`; exact cutoff re-applied after download |
| 2 | Float ID read from `'WMO'`, which argopy/erddap does not provide → **all IDs = 0** | Float ID from `PLATFORM_NUMBER` (hard error if absent) |
| 3 | Profiles grouped by `float_id + date`; with IDs all 0 this merged **all floats profiling on the same day** into one pseudo-profile, with GLORYS12 interpolated at the first float's position | Profiles grouped by `PLATFORM_NUMBER + CYCLE_NUMBER (+ DIRECTION)`; each profile checked for a single position and time |
| 4 | GLORYS12 matched to the monthly field with the **nearest timestamp** (can pick the next month if fields are stamped on the 1st) | Matched to the GLORYS12 field of the **same year and month** |

**Unchanged from the legacy method** (so figure changes reflect the fixes only): bounding box and Greenland Sea polygon;
argopy erddap source in `standard` mode; Argo in-situ T → potential T (TEOS-10, `gsw.pt_from_t`, p_ref = 0);
pressure (dbar) used as depth (configurable); bilinear horizontal interpolation of GLORYS12 `thetao`/`so`;
common depth grid (5 m to 100 m, 10 m to 500 m, 50 m to 2000 m); linear vertical interpolation, no extrapolation.

**New in the output**: real float IDs, cycle number, direction, the GLORYS12 file and timestamp used for each profile.

**Downstream**: point Figure S2 (`ARGO_FILE`) and Figure S5 (`ARGO_PAIRED_FILE`) at `OUT_PAIRED` below.
Variable names and dimensions (`T_argo`, `S_argo`, `T_glorys`, `S_glorys`, `float_id`, `lat`, `lon`, `time`;
dims `profile × depth`) match the legacy file, so no other changes are needed to run them.

## Configuration

In [5]:
import os, glob, json, logging, time as time_module
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import gsw
from matplotlib.path import Path as MplPath
from scipy.interpolate import interp1d

# --- Locked project conventions ---
DATA_END_DATE = '2026-04-30'                     # inclusive cutoff (end of winter 2026)
GS_POLYGON = [(-22, 71), (-8.5, 71), (12, 79), (-21, 79), (-28, 73), (-22, 71)]

# --- Argo download (as legacy) ---
YEAR_START = 1993                                # GLORYS12 start; no Argo here before ~2001
LON_MIN, LON_MAX = -28.0, 12.0                   # bounding box of the GS polygon
LAT_MIN, LAT_MAX = 71.0, 79.0
PRES_MIN, PRES_MAX = 0.0, 2000.0
ARGO_SOURCES = ['erddap', 'gdac']                # tried in order
ARGO_MODE = 'standard'                           # argopy user mode (QC-filtered, adjusted where available)

# --- GLORYS12 (monthly fields) ---
GLORYS_GLOB = '../../glorys12_with_density/glorys12_*_Greenland_Sea_with_density.nc'
GLORYS_TEMP, GLORYS_SAL = 'thetao', 'so'
GLORYS_LAT, GLORYS_LON, GLORYS_DEPTH, GLORYS_TIME = 'latitude', 'longitude', 'depth', 'time'

# --- Pairing method (as legacy) ---
COMMON_DEPTHS = np.concatenate([np.arange(0, 100, 5),
                                np.arange(100, 500, 10),
                                np.arange(500, 2050, 50)]).astype(float)
PRESSURE_TO_DEPTH = False   # False = legacy (pressure in dbar used as depth in m);
                            # True = convert with gsw.z_from_p (changes depths by <1% in the upper 2000 m)
MIN_LEVELS_PER_PROFILE = 2  # legacy: profiles with <2 valid levels give all-NaN

# --- Reprocessing control ---
REDOWNLOAD = False          # True: re-fetch yearly Argo files even if cached
REPROCESS  = True           # True: rebuild profile table and paired file

# --- Paths ---
OUT_DIR       = Path('./outputs/processed_data/argo')
RAW_DIR       = OUT_DIR / 'raw_yearly'
OUT_PROFILES  = OUT_DIR / 'argo_profiles_greenland_sea.nc'
OUT_PAIRED    = OUT_DIR / 'argo_glorys_paired_profiles_v2.nc'
OUT_COVERAGE  = OUT_DIR / 'argo_pairing_coverage_by_year_month.csv'
OUT_FLOATS    = OUT_DIR / 'argo_pairing_floats_by_year.csv'
METHODS_FILE  = Path('./outputs/methods/argo_glorys_pairing_methods.md')
LEGACY_PAIRED = Path('/local/jbj13rpu/Documents/ROVER/code/argo_glorys_paired_profiles.nc')  # for comparison only
LOG_DIR       = Path('./outputs/logs')

for d in [RAW_DIR, METHODS_FILE.parent, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

END_TS = pd.Timestamp(DATA_END_DATE) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)  # 23:59:59 on cutoff day
YEAR_END = pd.Timestamp(DATA_END_DATE).year

logger = logging.getLogger('argo_glorys_pairing')
logger.handlers.clear()
logger.setLevel(logging.INFO)
_fmt = logging.Formatter('%(asctime)s %(levelname)s %(message)s', '%H:%M:%S')
for h in [logging.StreamHandler(),
          logging.FileHandler(LOG_DIR / f"argo_glorys_pairing_{datetime.now():%Y%m%d_%H%M%S}.log")]:
    h.setFormatter(_fmt)
    logger.addHandler(h)
logger.info(f"Cutoff: {END_TS}  |  years {YEAR_START}-{YEAR_END}  |  REDOWNLOAD={REDOWNLOAD} REPROCESS={REPROCESS}")

22:39:10 INFO Cutoff: 2026-04-30 23:59:59  |  years 1993-2026  |  REDOWNLOAD=False REPROCESS=True


## Step 1 — Download Argo by year (fix 1: December and cutoff)

Each year is requested as `[YYYY-01-01, (YYYY+1)-01-01)`; the final year stops at the day after `DATA_END_DATE`.
Files are cached per year in `RAW_DIR`; set `REDOWNLOAD = True` to refresh.

Two download routes (`ARGO_BACKEND`):

- `'erddap_csv'` (default): queries the same Ifremer ERDDAP dataset that argopy uses (`ArgoFloats`), but as CSV, so no
  netCDF time decoding is involved. argopy's `standard` mode is then applied here: adjusted values for data modes A/D,
  real-time values for R, and only points whose position, time, pressure, temperature and salinity QC flags are 1 or 2.
- `'argopy'`: the argopy `DataFetcher` route, as in the legacy pipeline.

A year is marked empty only if the server reports no data. Any other error stops the run (`STOP_ON_FETCH_ERROR`).

In [6]:
from urllib.parse import quote
import urllib.request, urllib.error, io

ARGO_BACKEND = 'erddap_csv'       # 'erddap_csv' or 'argopy'
ERDDAP_URL = 'https://erddap.ifremer.fr/erddap/tabledap/ArgoFloats.csv'
STOP_ON_FETCH_ERROR = True        # False: skip failing years (not marked empty) and continue
QC_KEEP = [1, 2]                  # argopy standard mode


class NoArgoData(Exception):
    pass


def year_window(year):
    start = pd.Timestamp(f'{year}-01-01')
    end = min(pd.Timestamp(f'{year + 1}-01-01'), pd.Timestamp(DATA_END_DATE) + pd.Timedelta(days=1))
    return start, end


# ---------- Route 1: ERDDAP CSV (no netCDF decoding) ----------
ERDDAP_VARS = ['platform_number', 'cycle_number', 'direction', 'data_mode',
               'time', 'time_qc', 'latitude', 'longitude', 'position_qc'] + \
              [f'{v}{s}' for v in ['pres', 'temp', 'psal'] for s in ['', '_qc', '_adjusted', '_adjusted_qc']]


def erddap_query(start, end):
    cons = [f'longitude>={LON_MIN}', f'longitude<={LON_MAX}', f'latitude>={LAT_MIN}', f'latitude<={LAT_MAX}',
            f'pres>={PRES_MIN}', f'pres<={PRES_MAX}',
            f'time>={start:%Y-%m-%dT%H:%M:%SZ}', f'time<{end:%Y-%m-%dT%H:%M:%SZ}']
    return ERDDAP_URL + '?' + ','.join(ERDDAP_VARS) + ''.join('&' + quote(c, safe='=:.-') for c in cons)


def read_erddap_csv(url, retries=3):
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=300) as r:
                return pd.read_csv(io.BytesIO(r.read()), skiprows=[1], low_memory=False)  # row 2 = units
        except urllib.error.HTTPError as e:
            body = e.read().decode(errors='ignore')
            if e.code == 404 and 'no matching results' in body.lower():
                raise NoArgoData(body[:200])
            if attempt == retries - 1:
                raise RuntimeError(f"HTTP {e.code}: {body[:300]}")
        except urllib.error.URLError as e:
            if attempt == retries - 1:
                raise
        time_module.sleep(10 * (attempt + 1))


def apply_standard_mode(df):
    # argopy 'standard' mode: adjusted values for data mode A/D, raw for R; keep QC in QC_KEEP for all fields
    df = df.copy()
    dm = df['data_mode'].astype(str).str.strip()
    use_adj = dm.isin(['A', 'D'])
    for v in ['pres', 'temp', 'psal']:
        df[v.upper()] = np.where(use_adj, df[f'{v}_adjusted'], df[v]).astype(float)
        df[f'{v.upper()}_QC'] = pd.to_numeric(np.where(use_adj, df[f'{v}_adjusted_qc'], df[f'{v}_qc']),
                                              errors='coerce')
    for q in ['position_qc', 'time_qc']:
        df[q.upper()] = pd.to_numeric(df[q], errors='coerce')
    keep = np.ones(len(df), bool)
    for q in ['POSITION_QC', 'TIME_QC', 'PRES_QC', 'TEMP_QC', 'PSAL_QC']:
        keep &= df[q].isin(QC_KEEP).to_numpy()
    out = pd.DataFrame({
        'PLATFORM_NUMBER': pd.to_numeric(df['platform_number'].astype(str).str.strip(), errors='coerce'),
        'CYCLE_NUMBER': pd.to_numeric(df['cycle_number'], errors='coerce'),
        'DIRECTION': df['direction'].astype(str).str.strip(),
        'DATA_MODE': dm,
        'TIME': pd.to_datetime(df['time'], utc=True).dt.tz_localize(None),
        'LATITUDE': df['latitude'].astype(float), 'LONGITUDE': df['longitude'].astype(float),
        'PRES': df['PRES'], 'TEMP': df['TEMP'], 'PSAL': df['PSAL']})[keep]
    return out.dropna(subset=['PRES']).reset_index(drop=True), int((~keep).sum())


def fetch_year_erddap_csv(year):
    # month by month to keep responses small; returns (Dataset, info) or raises NoArgoData
    start, end = year_window(year)
    frames, n_qc = [], 0
    for m0 in pd.date_range(start, end, freq='MS', inclusive='left'):
        m1 = min(m0 + pd.offsets.MonthBegin(1), end)
        try:
            raw = read_erddap_csv(erddap_query(m0, m1))
        except NoArgoData:
            continue
        df, nbad = apply_standard_mode(raw)
        n_qc += nbad
        frames.append(df)
    if not frames:
        raise NoArgoData(f'no data {start:%Y-%m-%d} to {end:%Y-%m-%d}')
    df = pd.concat(frames, ignore_index=True)
    ds = xr.Dataset({c: ('N_POINTS', df[c].values) for c in df.columns})
    ds.attrs.update({'source': ERDDAP_URL, 'backend': 'erddap_csv', 'mode': 'standard (applied locally)',
                     'n_points_removed_by_qc': n_qc,
                     'download_date': datetime.now().isoformat(timespec='seconds')})
    return ds, f'erddap_csv ({n_qc} points failed QC)'


# ---------- Route 2: argopy (legacy route) ----------
def fetch_year_argopy(year):
    import argopy
    from argopy import DataFetcher
    argopy.set_options(mode=ARGO_MODE)
    start, end = year_window(year)
    box = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX, PRES_MIN, PRES_MAX,
           start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')]
    errors = {}
    for src in ARGO_SOURCES:
        try:
            ds = DataFetcher(src=src).region(box).load().data
            ds.attrs.update({'argopy_version': argopy.__version__, 'argopy_source': src, 'argopy_mode': ARGO_MODE})
            return ds, src
        except Exception as e:
            errors[src] = f"{type(e).__name__}: {e}"
    if all(e.split(':')[0] in ('DataNotFound', 'FileNotFoundError') for e in errors.values()):
        raise NoArgoData(json.dumps(errors))
    raise RuntimeError(json.dumps(errors, indent=1))


fetch_year = fetch_year_erddap_csv if ARGO_BACKEND == 'erddap_csv' else fetch_year_argopy

download_log = []
for year in range(YEAR_START, YEAR_END + 1):
    fpath = RAW_DIR / f'argo_raw_{year}.nc'
    empty_flag = RAW_DIR / f'argo_raw_{year}.empty'   # year confirmed to have no data
    if (fpath.exists() or empty_flag.exists()) and not REDOWNLOAD:
        download_log.append(dict(year=year, status='cached' if fpath.exists() else 'empty (cached)'))
        continue
    logger.info(f"Downloading {year} via {ARGO_BACKEND} ...")
    try:
        ds_y, info = fetch_year(year)
    except NoArgoData as e:
        empty_flag.write_text(str(e))
        logger.info(f"  {year}: no data")
        download_log.append(dict(year=year, status='empty'))
        continue
    except Exception as e:
        if STOP_ON_FETCH_ERROR:
            raise RuntimeError(f"Argo download failed for {year}: {type(e).__name__}: {e}") from e
        logger.warning(f"  {year}: download FAILED, skipped (not marked empty): {type(e).__name__}: {e}")
        download_log.append(dict(year=year, status='FAILED'))
        continue
    for v in ds_y.variables:                     # object attrs can break to_netcdf
        ds_y[v].attrs = {k: (str(a) if not isinstance(a, (str, int, float)) else a)
                         for k, a in ds_y[v].attrs.items()}
    ds_y.to_netcdf(fpath)
    logger.info(f"  {year}: {ds_y.sizes['N_POINTS']} points, {info} -> {fpath.name}")
    download_log.append(dict(year=year, status=f'downloaded ({info})'))

pd.DataFrame(download_log).to_csv(OUT_DIR / 'argo_download_log.csv', index=False)
failed = [r['year'] for r in download_log if r['status'] == 'FAILED']
if failed:
    logger.warning(f"Years that failed and were skipped: {failed}")
logger.info(f"Yearly files available: {len(sorted(RAW_DIR.glob('argo_raw_*.nc')))}")

22:39:10 INFO Downloading 1995 via erddap_csv ...
22:39:37 INFO   1995: no data
22:39:37 INFO Downloading 1996 via erddap_csv ...
22:40:03 INFO   1996: no data
22:40:03 INFO Downloading 1997 via erddap_csv ...
22:40:33 INFO   1997: no data
22:40:33 INFO Downloading 1998 via erddap_csv ...
22:41:04 INFO   1998: no data
22:41:04 INFO Downloading 1999 via erddap_csv ...
22:41:40 INFO   1999: no data
22:41:40 INFO Downloading 2000 via erddap_csv ...
22:42:12 INFO   2000: no data
22:42:12 INFO Downloading 2003 via erddap_csv ...
22:42:26 INFO   2003: 4252 points, erddap_csv (644 points failed QC) -> argo_raw_2003.nc
22:42:26 INFO Downloading 2004 via erddap_csv ...
22:42:45 INFO   2004: 3349 points, erddap_csv (2 points failed QC) -> argo_raw_2004.nc
22:42:45 INFO Downloading 2005 via erddap_csv ...
22:42:56 INFO   2005: 16963 points, erddap_csv (10 points failed QC) -> argo_raw_2005.nc
22:42:56 INFO Downloading 2006 via erddap_csv ...
22:43:06 INFO   2006: 25839 points, erddap_csv (54 poin

## Step 2 — Build a profile table (fixes 2 and 3: real float IDs, one profile per float cycle)

argopy/erddap returns a flat point cloud (`N_POINTS`). Points are masked to the Greenland Sea polygon, restricted to the
cutoff, de-duplicated, and grouped by `PLATFORM_NUMBER + CYCLE_NUMBER (+ DIRECTION)`. Every profile is checked to have a
single position and time; any that do not are logged and dropped rather than silently merged.

In [7]:
KEEP_VARS = ['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION', 'TIME', 'LATITUDE', 'LONGITUDE', 'PRES', 'TEMP', 'PSAL']


def in_polygon(lon, lat, poly=GS_POLYGON):
    return MplPath(np.array(poly)).contains_points(np.column_stack([lon, lat]))


def load_points(files):
    frames = []
    for f in files:
        ds = xr.open_dataset(f)
        missing = [v for v in ['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'TIME', 'LATITUDE', 'LONGITUDE',
                               'PRES', 'TEMP', 'PSAL'] if v not in ds.variables]
        if missing:
            raise KeyError(f"{f.name}: missing {missing}; variables are {list(ds.variables)}")
        df = ds[[v for v in KEEP_VARS if v in ds.variables]].to_dataframe().reset_index(drop=True)
        ds.close()
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    if 'DIRECTION' not in df:
        df['DIRECTION'] = 'A'
    df['PLATFORM_NUMBER'] = pd.to_numeric(df['PLATFORM_NUMBER'].astype(str).str.strip(), errors='coerce')
    df['DIRECTION'] = df['DIRECTION'].astype(str).str.strip()
    df['TIME'] = pd.to_datetime(df['TIME'])
    return df


if REPROCESS or not OUT_PROFILES.exists():
    files = sorted(RAW_DIR.glob('argo_raw_*.nc'))
    pts = load_points(files)
    n0 = len(pts)
    pts = pts[(pts['TIME'] <= END_TS)]
    pts = pts[in_polygon(pts['LONGITUDE'].values, pts['LATITUDE'].values)]
    pts = pts.dropna(subset=['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'PRES'])
    pts = pts.drop_duplicates(subset=['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION', 'TIME', 'PRES'])
    logger.info(f"Points: {n0} downloaded -> {len(pts)} in polygon, <= cutoff, de-duplicated")
    if (pts['PLATFORM_NUMBER'] == 0).any():
        raise ValueError("PLATFORM_NUMBER == 0 found: float IDs not read correctly")

    keys = ['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION']
    g = pts.groupby(keys, sort=False)
    chk = g.agg(n_lat=('LATITUDE', 'nunique'), n_lon=('LONGITUDE', 'nunique'), n_time=('TIME', 'nunique'))
    bad = chk[(chk['n_lat'] > 1) | (chk['n_lon'] > 1) | (chk['n_time'] > 1)]
    logger.info(f"Profiles: {len(chk)}  |  with inconsistent position/time (dropped): {len(bad)}")
    if len(bad):
        pts = pts.set_index(keys).drop(index=bad.index).reset_index()

    profiles = []
    for (fid, cyc, dirn), grp in pts.groupby(keys, sort=False):
        grp = grp.sort_values('PRES')
        profiles.append(dict(float_id=int(fid), cycle=int(cyc), direction=dirn,
                             time=grp['TIME'].iloc[0], lat=float(grp['LATITUDE'].iloc[0]),
                             lon=float(grp['LONGITUDE'].iloc[0]),
                             pres=grp['PRES'].to_numpy(float), temp=grp['TEMP'].to_numpy(float),
                             sal=grp['PSAL'].to_numpy(float)))
    profiles.sort(key=lambda p: p['time'])
    logger.info(f"Profile table: {len(profiles)} profiles from {len({p['float_id'] for p in profiles})} floats, "
                f"{profiles[0]['time']:%Y-%m-%d} to {profiles[-1]['time']:%Y-%m-%d}")
else:
    raise RuntimeError("REPROCESS=False: set REPROCESS=True to rebuild (profile table is not cached as a table).")

22:51:47 INFO Points: 5553969 downloaded -> 2448666 in polygon, <= cutoff, de-duplicated
22:51:47 INFO Profiles: 6816  |  with inconsistent position/time (dropped): 0
22:51:51 INFO Profile table: 6816 profiles from 152 floats, 2001-04-25 to 2026-04-30


## Step 3 — Index GLORYS12 by year and month (fix 4)

Each profile is matched to the GLORYS12 field for the **same year and month**. The file and timestamp used are recorded
per profile, so any product/stream boundary (e.g. reprocessed vs interim) can be identified downstream.

In [8]:
glorys_files = sorted(glob.glob(GLORYS_GLOB))
if not glorys_files:
    raise FileNotFoundError(f"No GLORYS12 files match {GLORYS_GLOB}")

gindex = {}   # (year, month) -> (file, time index, timestamp)
for f in glorys_files:
    with xr.open_dataset(f) as ds:
        tt = pd.to_datetime(ds[GLORYS_TIME].values)
    for i, t in enumerate(tt):
        key = (t.year, t.month)
        if key in gindex:
            raise ValueError(f"More than one GLORYS12 field for {key} ({gindex[key][0]} and {f}); "
                             "expected monthly fields.")
        gindex[key] = (f, i, t)

gmonths = sorted(gindex)
logger.info(f"GLORYS12: {len(glorys_files)} files, {len(gindex)} monthly fields, "
            f"{gmonths[0][0]}-{gmonths[0][1]:02d} to {gmonths[-1][0]}-{gmonths[-1][1]:02d}")
logger.info(f"GLORYS12 timestamp day-of-month values: {sorted({v[2].day for v in gindex.values()})}")

22:52:34 INFO GLORYS12: 34 files, 401 monthly fields, 1993-01 to 2026-05
22:52:34 INFO GLORYS12 timestamp day-of-month values: [1]


## Step 4 — Pair profiles with GLORYS12

Profiles are processed month by month: each GLORYS12 monthly field is loaded once and interpolated bilinearly to all
profile positions in that month. Argo in-situ temperature is converted to potential temperature before vertical
interpolation; both profiles are then interpolated linearly onto `COMMON_DEPTHS` without extrapolation.

In [9]:
def to_common_grid(z, v):
    ok = np.isfinite(z) & np.isfinite(v)
    if ok.sum() < MIN_LEVELS_PER_PROFILE:
        return np.full(COMMON_DEPTHS.shape, np.nan)
    z, v = z[ok], v[ok]
    o = np.argsort(z)
    z, v = z[o], v[o]
    z, iu = np.unique(z, return_index=True)       # guard against repeated levels
    v = v[iu]
    if len(z) < MIN_LEVELS_PER_PROFILE:
        return np.full(COMMON_DEPTHS.shape, np.nan)
    return interp1d(z, v, bounds_error=False, fill_value=np.nan)(COMMON_DEPTHS)


def argo_pt(p):
    t, s, pr = p['temp'], p['sal'], p['pres']
    out = np.full(t.shape, np.nan)
    ok = np.isfinite(t) & np.isfinite(s) & np.isfinite(pr)
    if ok.any():
        SA = gsw.SA_from_SP(s[ok], pr[ok], p['lon'], p['lat'])
        out[ok] = gsw.pt_from_t(SA, t[ok], pr[ok], 0)
    return out


def argo_depth(p):
    return -gsw.z_from_p(p['pres'], p['lat']) if PRESSURE_TO_DEPTH else p['pres']


t0 = time_module.time()
by_month = {}
for k, p in enumerate(profiles):
    by_month.setdefault((p['time'].year, p['time'].month), []).append(k)

rows, TA, SA_, TG, SG = [], [], [], [], []
skipped = {'no_glorys_month': 0, 'glorys_all_nan': 0, 'argo_all_nan': 0}
open_file, ds_open = None, None
for ym in sorted(by_month):
    idxs = by_month[ym]
    if ym not in gindex:
        skipped['no_glorys_month'] += len(idxs)
        continue
    f, ti, gtime = gindex[ym]
    if f != open_file:
        if ds_open is not None:
            ds_open.close()
        ds_open, open_file = xr.open_dataset(f), f
    field = ds_open[[GLORYS_TEMP, GLORYS_SAL]].isel({GLORYS_TIME: ti}).load()
    lats = xr.DataArray([profiles[k]['lat'] for k in idxs], dims='prof')
    lons = xr.DataArray([profiles[k]['lon'] for k in idxs], dims='prof')
    pt_g = field.interp({GLORYS_LAT: lats, GLORYS_LON: lons}, method='linear')
    gdep = field[GLORYS_DEPTH].values.astype(float)
    for j, k in enumerate(idxs):
        p = profiles[k]
        tg = to_common_grid(gdep, pt_g[GLORYS_TEMP].isel(prof=j).values.astype(float))
        sg = to_common_grid(gdep, pt_g[GLORYS_SAL].isel(prof=j).values.astype(float))
        z = argo_depth(p)
        ta = to_common_grid(z, argo_pt(p))
        sa = to_common_grid(z, p['sal'])
        if np.all(np.isnan(tg)) and np.all(np.isnan(sg)):
            skipped['glorys_all_nan'] += 1
            continue
        if np.all(np.isnan(ta)) and np.all(np.isnan(sa)):
            skipped['argo_all_nan'] += 1
            continue
        TA.append(ta); SA_.append(sa); TG.append(tg); SG.append(sg)
        rows.append(dict(float_id=p['float_id'], cycle=p['cycle'], direction=p['direction'],
                         lat=p['lat'], lon=p['lon'], time=p['time'],
                         glorys_file=os.path.basename(f), glorys_time=gtime,
                         n_argo_levels=int(np.isfinite(sa).sum())))
if ds_open is not None:
    ds_open.close()

meta = pd.DataFrame(rows)
logger.info(f"Paired {len(meta)} of {len(profiles)} profiles in {(time_module.time() - t0) / 60:.1f} min; "
            f"skipped: {skipped}")

22:53:09 INFO Paired 6809 of 6816 profiles in 0.5 min; skipped: {'no_glorys_month': 0, 'glorys_all_nan': 0, 'argo_all_nan': 7}


## Step 5 — Save the paired file

In [10]:
ds_out = xr.Dataset(
    {'T_argo':   (['profile', 'depth'], np.array(TA)),
     'S_argo':   (['profile', 'depth'], np.array(SA_)),
     'T_glorys': (['profile', 'depth'], np.array(TG)),
     'S_glorys': (['profile', 'depth'], np.array(SG)),
     'float_id': (['profile'], meta['float_id'].values.astype(np.int64)),
     'cycle':    (['profile'], meta['cycle'].values.astype(np.int64)),
     'direction': (['profile'], meta['direction'].values.astype(str)),
     'lat':      (['profile'], meta['lat'].values),
     'lon':      (['profile'], meta['lon'].values),
     'time':     (['profile'], pd.to_datetime(meta['time']).values),
     'glorys_file': (['profile'], meta['glorys_file'].values.astype(str)),
     'glorys_time': (['profile'], pd.to_datetime(meta['glorys_time']).values),
     'n_argo_levels': (['profile'], meta['n_argo_levels'].values)},
    coords={'depth': COMMON_DEPTHS})

ds_out['T_argo'].attrs = {'long_name': 'Argo potential temperature (p_ref = 0 dbar, TEOS-10)', 'units': 'degC'}
ds_out['T_glorys'].attrs = {'long_name': 'GLORYS12 thetao, bilinear to profile position', 'units': 'degC'}
ds_out['S_argo'].attrs = {'long_name': 'Argo practical salinity', 'units': 'psu'}
ds_out['S_glorys'].attrs = {'long_name': 'GLORYS12 so, bilinear to profile position', 'units': 'psu'}
ds_out['float_id'].attrs = {'long_name': 'Argo PLATFORM_NUMBER (WMO float ID)'}
ds_out['depth'].attrs = {'long_name': 'common depth grid', 'units': 'm',
                         'note': 'Argo pressure (dbar) used as depth' if not PRESSURE_TO_DEPTH
                                 else 'Argo depth from gsw.z_from_p'}
ds_out.attrs = {
    'title': 'Argo vs GLORYS12 paired T/S profiles, Greenland Sea',
    'created': datetime.now().isoformat(timespec='seconds'),
    'data_end_date': DATA_END_DATE,
    'polygon': json.dumps(GS_POLYGON),
    'argo_mode': ARGO_MODE,
    'profile_definition': 'PLATFORM_NUMBER + CYCLE_NUMBER + DIRECTION',
    'glorys_matching': 'same year and month; bilinear horizontal interpolation',
    'glorys_glob': GLORYS_GLOB,
    'n_profiles_paired': len(meta),
    'n_skipped': json.dumps(skipped),
    'supersedes': 'argo_glorys_paired_profiles.nc (legacy; December missing, float IDs zero, same-day floats merged)'}
enc = {v: {'zlib': True, 'complevel': 4} for v in ['T_argo', 'S_argo', 'T_glorys', 'S_glorys']}
ds_out.to_netcdf(OUT_PAIRED, encoding=enc)
logger.info(f"Saved {OUT_PAIRED} ({OUT_PAIRED.stat().st_size / 1e6:.1f} MB): "
            f"{len(meta)} profiles x {len(COMMON_DEPTHS)} levels")

22:53:25 INFO Saved outputs/processed_data/argo/argo_glorys_paired_profiles_v2.nc (14.7 MB): 6809 profiles x 91 levels


## Step 6 — Checks

Confirms the fixes: December present, float IDs non-zero, several floats per year, and which GLORYS12 files were
used in 2025–2026. If the legacy file is available, profile counts per year are compared with it.

In [11]:
m = meta.copy()
m['year'], m['month'] = m['time'].dt.year, m['time'].dt.month
cov = m.pivot_table(index='year', columns='month', values='float_id', aggfunc='size', fill_value=0)
cov = cov.reindex(columns=range(1, 13), fill_value=0)
cov.to_csv(OUT_COVERAGE)
print("Paired profiles per year and month:\n", cov.tail(12).to_string())

floats = m.groupby('year').agg(n_profiles=('float_id', 'size'), n_floats=('float_id', 'nunique'))
floats.to_csv(OUT_FLOATS)
print("\nProfiles and floats per year:\n", floats.to_string())

assert cov[12].sum() > 0, "No December profiles — check download windows"
assert (m['float_id'] > 0).all(), "Float IDs not populated"
assert m['month'].eq(m['glorys_time'].dt.month).all() and m['year'].eq(m['glorys_time'].dt.year).all(), \
    "Profile and GLORYS12 year-month do not match"

print("\nGLORYS12 files used, 2025 onwards:\n",
      m[m['year'] >= 2025].groupby('glorys_file').agg(first=('time', 'min'), last=('time', 'max'),
                                                     n=('time', 'size')).to_string())

LEGACY_RAW = LEGACY_PAIRED.parent / 'argo_greenland_sea_raw.nc'
if LEGACY_RAW.exists():   # validates the download route: points per year, Jan-Nov only (legacy lacks December)
    with xr.open_dataset(LEGACY_RAW) as old:
        t_old = pd.to_datetime(old['TIME'].values)
    new_t = pd.concat([pd.Series(pd.to_datetime(xr.open_dataset(f)['TIME'].values))
                       for f in sorted(RAW_DIR.glob('argo_raw_*.nc'))])
    cmp_raw = pd.DataFrame({'legacy_points': pd.Series(t_old[t_old.month < 12].year).value_counts(),
                            'new_points': new_t[new_t.dt.month < 12].dt.year.value_counts()}).fillna(0).astype(int)
    print("\nRaw points per year, Jan-Nov (bounding box, before polygon):\n", cmp_raw.to_string())

if LEGACY_PAIRED.exists():
    with xr.open_dataset(LEGACY_PAIRED) as old:
        old_years = pd.Series(pd.to_datetime(old['time'].values).year).value_counts().sort_index()
    comp = pd.DataFrame({'legacy': old_years, 'new': floats['n_profiles']}).fillna(0).astype(int)
    comp['change'] = comp['new'] - comp['legacy']
    print("\nProfiles per year, legacy vs new (legacy is before polygon filtering in S2/S5):\n", comp.to_string())

Paired profiles per year and month:
 month  1   2   3   4   5   6   7   8   9   10  11  12
year                                                 
2015   45  43  48  53  60  56  49  43  34  34  35  35
2016   17  15  15  16  24  23  19  28  33  35  36  35
2017   40  29  34  28  28  26  27  24  35  32  48  32
2018   28  31  36  16  16  24  18  30  25  20  11  10
2019    9   9  10  11  15  30  27  36  32  38  32  49
2020   35  31  25  19  23  30  35  35  32  36  35  25
2021   24  24  20  26  43  31  40  37  36  32  33  27
2022   21  21  24  19  15  39  45  43  54  42  32  33
2023   43  35  22  27  28  36  39  32  67  31  19  25
2024   21  19  19  12  12  17  10   7  14  13  16  16
2025   16  23  36  42  40  48  50  63  60  50  33  26
2026   20  20  35  40   0   0   0   0   0   0   0   0

Profiles and floats per year:
       n_profiles  n_floats
year                      
2001          82         5
2002         160         6
2003          60         6
2004          36         5
2005         

## Step 7 — Methods note

In [12]:
d = meta[meta['time'].dt.month < 12]
days = d.groupby(d['time'].dt.year)['time'].apply(lambda t: t.dt.normalize().nunique())
with xr.open_dataset(LEGACY_PAIRED) as old:
    t_old = pd.Series(pd.to_datetime(old['time'].values))
leg = t_old[t_old.dt.month < 12].dt.year.value_counts().sort_index()
print(pd.DataFrame({'legacy_profiles': leg, 'new_distinct_days': days}).fillna(0).astype(int).to_string())

      legacy_profiles  new_distinct_days
2001               47                 47
2002              106                106
2003               42                 41
2004               10                 10
2005               64                 64
2006               66                 66
2007              100                100
2008              146                146
2009               98                 98
2010               91                 91
2011              137                137
2012              163                163
2013              212                212
2014              191                191
2015              252                252
2016              177                177
2017              221                221
2018              182                181
2019              171                171
2020              215                215
2021              197                197
2022              208                208
2023              160                160
2024            

In [ ]:
lines = [
    "# Argo–GLORYS12 pairing (data prep for Figures S2 and S5)", "",
    f"- Argo profiles downloaded from the Ifremer ERDDAP Argo dataset "
    f"({'as CSV with argopy standard-mode filtering applied locally' if ARGO_BACKEND == 'erddap_csv' else 'via argopy'}: "
    "adjusted values for delayed/adjusted modes, QC flags 1-2 only) for the box "
    f"{LON_MIN}–{LON_MAX}°E, {LAT_MIN}–{LAT_MAX}°N, {PRES_MIN:.0f}–{PRES_MAX:.0f} dbar, in yearly windows "
    f"[1 Jan, 1 Jan next year), up to {DATA_END_DATE} inclusive; restricted to the Greenland Sea polygon.",
    "- A profile is one PLATFORM_NUMBER + CYCLE_NUMBER (+ DIRECTION); profiles with more than one position or time "
    f"are dropped ({len(bad)} dropped).",
    "- Argo in-situ temperature converted to potential temperature (TEOS-10, gsw.pt_from_t, p_ref = 0 dbar) "
    "for comparison with GLORYS12 thetao.",
    "- GLORYS12 monthly thetao and so matched to each profile by year and month, interpolated bilinearly to the "
    "profile position. Both profiles interpolated linearly onto a common depth grid (5 m to 100 m, 10 m to 500 m, "
    "50 m to 2000 m) without extrapolation; " +
    ("Argo pressure (dbar) used as depth (m)." if not PRESSURE_TO_DEPTH else "Argo depth from gsw.z_from_p."),
    f"- Result: {len(meta)} paired profiles from {m['float_id'].nunique()} floats, "
    f"{m['time'].min():%Y-%m-%d} to {m['time'].max():%Y-%m-%d}; skipped {skipped}.",
    "", "## Changes from the legacy pairing (argo_glorys_paired_profiles.nc)",
    "- December was missing in all years (exclusive 'YYYY-12' download bound). Fixed.",
    "- Float IDs were all 0 (wrong variable name), so all floats profiling on the same day were merged into one "
    "pseudo-profile with GLORYS12 interpolated at the first float's position. Fixed.",
    "- GLORYS12 field chosen by nearest timestamp; now by same year and month.",
    "", "## Notes",
    "- The GLORYS12 file used for each profile is stored (glorys_file); check for product/stream changes near the "
    "end of the record.",
    "- Monthly-mean GLORYS12 is compared with instantaneous profiles; mismatch includes sub-monthly variability, "
    "which is largest near fronts and the ice edge.",
]
METHODS_FILE.write_text("\n".join(lines) + "\n")
logger.info(f"Methods written to {METHODS_FILE}")